# Chapter 18 &mdash; Beta Reduction, Watched

**Concept 12 of the Chapter 18 decomposition:** *Beta Reduction, Watched: Substitution and the Capture It Avoids*

The rule of Concept 3, actually firing &mdash; the bound variable in red, the argument in blue, and the renaming that stops a free name being swallowed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-Beta-Reduction-Watched/Concept-Beta-Reduction-Watched.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateLambda  import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateLambda as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateLambda, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Concept 3 states the rule:

$$(\lambda x.\,M)\;N \;\to\; M[x := N]$$

and shows nothing doing it. Here it fires, one step at a time.

**Colour says what is about to happen.** In the boxed redex the **binder and every
bound occurrence** are red and the **argument** is blue. Before pressing step you can
see which letters are about to be replaced and by what; afterwards you can count the
copies that landed &mdash; one per occurrence, **none** if the variable was unused,
**several** if it appeared several times. Substitution is duplication, and that is
visible rather than described.

**And then capture.** Substituting naively into

$$(\lambda x.\,\lambda y.\,x)\;y$$

would carry the free $y$ under the $\lambda y$ and bind it, turning a constant
function into the identity. That is not a subtlety; it is a wrong answer. Real
substitution renames the binder first, and the animation **announces the renaming when
it fires** &mdash; because a side condition you are told about and never see trigger is
a side condition you do not believe.

**The receipt for Concept 11.** `PLUS 2 3` reduces here in **8** beta steps. The same
arithmetic in SKI took **108**. Bracket abstraction did not make anything faster; it
compiled substitution away into copying, and the two numbers are the price.

## 2. Definitions

### Terms, and the one rule

In [ ]:
from jove.AnimateLambda import *

t = parse(r'(\x. x) y')
print('term      :', show(t))
nf, n = reduce_lam(t)
print('reduces to:', show(nf), ' in', n, 'step')
print()
print(r"Use \ for lambda.  parse(r'\x.\y. x') builds a term; show() prints one.")
print('Application is left-associative, so  f a b  is  (f a) b.')

### Substitution duplicates, drops, or does nothing

In [ ]:
cases = [(r'(\x. x x) a',        'x occurs twice'),
         (r'(\x. y) a',          'x does not occur at all'),
         (r'(\x. x) a',          'x occurs once'),
         (r'(\f.\x. f (f x)) g z', 'a Church two, applied')]
for src, why in cases:
    nf, n = reduce_lam(parse(src))
    print('  %-24s -> %-12s  %-22s (%d steps)'
          % (src.replace('\\', '\\'), show(nf), why, n))
print()
print('Count the copies of the argument in each answer.  That is the whole')
print('content of the rule: as many copies as the variable had occurrences.')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch18&nbsp;11.&nbsp;S and K, Actually Running: Arithmetic and Recursion in SKI](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18-Lambda/Concept-SKI-Actually-Running/Concept-SKI-Actually-Running.ipynb) &nbsp;&middot;&nbsp; [**Chapter 18** index](https://github.com/ganeshutah/Jove/blob/master/Chapter18-Lambda/README.md)

---

## 3. Tests

**The capture case.** This is the one that matters.

In [ ]:
t = parse(r'(\x.\y. x) y')
print('term          :', show(t))
nf, _ = reduce_lam(t)
print('reduces to    :', show(nf))
print()
print('WRONG answer would be  ' + LAM + 'y.y  -- the identity.')
print('The free y would have walked under the binder and become bound.')
assert show(nf) != LAM + 'y.y'
print()
tr = trace_lam(t)
print('renaming that prevented it:', tr[0][4])
assert tr[0][4], 'the animation should report an alpha-renaming here'

Without the renaming the term means something else entirely &mdash; a constant function becomes the identity.

In [ ]:
const_y = reduce_lam(parse(r'(\x.\y. x) y'))[0]
ident   = parse(r'\y. y')
print('what we got  :', show(const_y), '  -- ignores its argument, returns y')
print('the wrong one:', show(ident), '   -- returns its argument')
print()
for a in ('p', 'q'):
    got = show(reduce_lam(Ap(const_y, V(a)))[0])
    print('  applied to %s :  %s' % (a, got))
print()
print('It answers y whatever you feed it.  The identity would not.')

**Arithmetic, reduced directly.** No combinators, no bracket abstraction.

In [ ]:
for m, n in ((2, 3), (0, 4), (3, 3)):
    got = unchurch(Ap(PLUS, church(m), church(n)))
    print('  %d + %d = %s' % (m, n, got))
    assert got == m + n
print('  3 x 4 =', unchurch(Ap(MULT, church(3), church(4))))
print('  succ 4 =', unchurch(Ap(SUCC, church(4))))
print()
nf, steps = reduce_lam(Ap(PLUS, church(2), church(3), V('f'), V('x')))
print('PLUS 2 3 f x  ->', show(nf), ' in', steps, 'beta steps')
assert show(nf) == 'f (f (f (f (f x))))'

**The comparison with Concept 11.** Same arithmetic, two calculi.

In [ ]:
print('%-22s %8s' % ('', 'steps'))
print('%-22s %8d' % ('beta reduction', steps))
print('%-22s %8d' % ('SKI (Concept 11)', 108))
print()
print('Bracket abstraction did not speed anything up.  It removed the need')
print('for substitution -- and therefore for capture-avoidance, and for the')
print('renaming above -- by compiling all of it into copying.  S x y z')
print('duplicates z; that is where the other hundred steps went.')

## 4. Animation

Step through it. The box is the redex, red is the variable about to be replaced,
blue is what replaces it. The default is `(\x.\y. x) a b`.

Try the capture case &mdash; `AnimateLambda(r'(\x.\y. x) y')` &mdash; and watch the
amber renaming notice appear. Or `AnimateLambda(Ap(PLUS, church(2), church(3), V('f'),
V('x')), 'PLUS 2 3 f x')` for the eight-step arithmetic.

In [ ]:
from jove.AnimateLambda import *
AnimateLambda()

## 5. Exercises


1. `(\x. x x) (\x. x x)` has no normal form. Predict what the animation does, then
   run it. What does the step limit protect you from?
2. In `PLUS 2 3` no renaming is needed. Construct a sum where one **is**, and check
   the notice appears.
3. Reduce `(\x. y) ((\x. x x) (\x. x x))` &mdash; an argument with no normal form,
   discarded. Why does leftmost-outermost order terminate where innermost would not?
4. Count the copies of the argument at each step of `(\x. x x x) a`. Relate the
   count to the number of occurrences of `x`.
5. `ISZERO`, `TRUE` and `FALSE` are provided. Build `IF (ISZERO 0) a b` and reduce it.
   Which branch is *never* touched, and why does that matter for recursion?
6. The capture case renamed `y` to `y1`. Would renaming to `z` be equally correct?
   What exactly is the requirement on the new name?
7. SKI needs 108 steps for what this does in 8, yet SKI is the simpler machine. State
   the trade precisely, in terms of what each calculus has to carry around.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter18-Lambda/Concept-Beta-Reduction-Watched')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')